# Distributed Training Puzzles

The goal of this notebook is to help you learn some distributed training algorithms by solving a series of hands-on puzzles. Specifically, you will:
* Implement different parallelism strategies that let model training scale across many devices;
* Get familiar with the [PyTorch Distributed](https://docs.pytorch.org/tutorials/intermediate/dist_tuto.html) framework;
* Implement your own collective operations from scratch.

For convenience, we'll use the [Gloo](https://github.com/pytorch/gloo) backend, which supports collective operations on CPU machines -- similar to how [NCCL](https://github.com/NVIDIA/nccl) supports them on GPUs. The best part: both backends use the same PyTorch Distributed API!

This means the notebook is designed to run on a CPU-only Google Colab runtime. All puzzles come with built-in tests -- you can check your solutions instantly! Reference cell outputs for successful solutions are provided for your convenience.

>[Puzzle A: Ring AllReduce (20 points)](#scrollTo=NA-sZITD2Ub4)

>[Puzzle B: Tensor Parallelism (20 points)](#scrollTo=cgvBRyQB2_ZW)

>[Puzzle C: Ulysses (20 points)](#scrollTo=GV0OhqawuJqf)

>[Puzzle D: Ring Attention (30 points)](#scrollTo=dxN7zsYfC65C)

>[Puzzle E: Zero Bubble Parallelism (10 points)](#scrollTo=bG2TD7UVoQvQ)


## Puzzle A: Ring AllReduce (20 points)

One of the foundational collective operations in distributed training is AllReduce: it allows you to sum the model gradients between the GPUs in the Distributed Data Parallel (DDP) training. On top of that, AllReduce is also used inside of different model parallelism strategies.

<img src="https://drive.google.com/uc?id=1u5b_SEdG4vY7VeCX7cQznsjVV4Lljntu" alt="allreduce.png">

See how AllReduce may be used in PyTorch Distributed:

In [1]:
%%writefile allreduce_example.py

import os
import torch
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh

def main():
    world_size = int(os.environ["WORLD_SIZE"])
    mesh = init_device_mesh("cpu", mesh_shape=(world_size,), mesh_dim_names=("dp",))

    rank = dist.get_rank()
    x = (torch.arange(4) + rank * 4).float()
    print(f"[rank {rank}] allreduce input: {x}", flush=True)

    pg = mesh.get_group("dp")
    dist.all_reduce(x, group=pg)
    print(f"[rank {rank}] allreduce output: {x}", flush=True)

if __name__ == "__main__":
    main()

Overwriting allreduce_example.py


In [2]:
!OMP_NUM_THREADS=1 torchrun --standalone --nnodes=1 --nproc-per-node=4 allreduce_example.py | grep -vF 'Gloo'

[rank 1] allreduce input: tensor([4., 5., 6., 7.])
[rank 0] allreduce input: tensor([0., 1., 2., 3.])
[rank 3] allreduce input: tensor([12., 13., 14., 15.])
[rank 2] allreduce input: tensor([ 8.,  9., 10., 11.])
[rank3]: Traceback (most recent call last):
[rank3]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/perf_enginnering_hw_2/allreduce_example.py", line 20, in <module>
[rank3]:     main()
[rank3]:     ~~~~^^
[rank3]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/perf_enginnering_hw_2/allreduce_example.py", line 16, in main
[rank3]:     dist.all_reduce(x, group=pg)
[rank3]:     ~~~~~~~~~~~~~~~^^^^^^^^^^^^^
[rank3]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/.venv/lib/python3.13/site-packages/torch/distributed/c10d_logger.py", line 83, in wrapper
[rank3]:     return func(*args, **kwargs)
[rank3]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/.venv/lib/python3.13/site-packages/torch/distributed/dis

> 💡 Check pytorch docs for the [Device Mesh](https://docs.pytorch.org/tutorials/recipes/distributed_device_mesh.html) and [torchrun](https://docs.pytorch.org/docs/stable/elastic/run.html) reference, or prompt your favourite LLM.

Let's implement the naive algorithm of AllReduce:
* Rank 0 gathers tensors from all ranks;
* Rank 0 sums them;
* And then rank 0 redistributes the result back to all tensors.

In [3]:
%%writefile allreduce_naive.py

import os
import torch
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh

def naive_allreduce(x: torch.Tensor, pg: dist.ProcessGroup):
    group_rank = dist.get_rank(group=pg)
    group_size = dist.get_world_size(group=pg)
    root = 0

    if group_rank == root:
        total = x.clone()
        for src in range(1, group_size):
            buf = torch.empty_like(x)
            dist.recv(buf, group=pg, group_src=src)
            total += buf

        x.copy_(total)

        for dst in range(1, group_size):
            dist.send(total, group=pg, group_dst=dst)
    else:
        dist.send(x, group=pg, group_dst=root)
        dist.recv(x, group=pg, group_src=root)

def main():
    world_size = int(os.environ["WORLD_SIZE"])
    mesh = init_device_mesh("cpu", mesh_shape=(world_size,), mesh_dim_names=("dp",))

    rank = dist.get_rank()
    x = (torch.arange(4) + rank * 4).float()
    print(f"[rank {rank}] allreduce input: {x}", flush=True)

    pg = mesh.get_group("dp")
    naive_allreduce(x, pg)

    print(f"[rank {rank}] allreduce output: {x}", flush=True)

if __name__ == "__main__":
    main()

Overwriting allreduce_naive.py


In [4]:
!OMP_NUM_THREADS=1 torchrun --standalone --nnodes=1 --nproc-per-node=4 allreduce_naive.py | grep -vF 'Gloo'

[rank 2] allreduce input: tensor([ 8.,  9., 10., 11.])[rank 0] allreduce input: tensor([0., 1., 2., 3.])

[rank 3] allreduce input: tensor([12., 13., 14., 15.])
[rank 1] allreduce input: tensor([4., 5., 6., 7.])
[rank3]: Traceback (most recent call last):
[rank3]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/perf_enginnering_hw_2/allreduce_naive.py", line 41, in <module>
[rank3]:     main()
[rank3]:     ~~~~^^
[rank3]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/perf_enginnering_hw_2/allreduce_naive.py", line 36, in main
[rank3]:     naive_allreduce(x, pg)
[rank3]:     ~~~~~~~~~~~~~~~^^^^^^^
[rank3]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/perf_enginnering_hw_2/allreduce_naive.py", line 24, in naive_allreduce
[rank3]:     dist.send(x, group=pg, group_dst=root)
[rank3]:     ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[rank3]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/.venv/lib/python3.13/site-pac

In the naive implementation provided above, `rank 0` becomes a bottleneck: its communication volume is $2s\left(n - 1\right)$, where $n$ is the world size, and $s$ is the number of elements in tensor. Thus, if you use this approach in the real training pipeline, you may notice that the time of AllReduce scales linearly with the number of processes (devices) $n$, which is inefficient.

In practice, people use more efficient AllReduce algorithms. Your goal in this puzzle is to implement [Ring AllReduce](https://www.cs.fsu.edu/~xyuan/paper/09jpdc.pdf), where each rank communicates only $2s\left(1-\frac{1}{n}\right)$ elements. That is the key bandwidth advantage: the load is now balanced across all ranks.

Your goal is to implement Ring Allreduce algorithm:

<img src="https://drive.google.com/uc?id=1YLEcOLhexatqbiIRN7jegIpR5fGCFaH_" alt="allreduce_steps.png">

Implement the algorithm in the cell below. Hints:
* You may assume `x` is contiguous and `x.numel()` is divisible by the number of processes;
* Use `dist.recv` and `dist.isend` for p2p communications *(do not use `batch_isend_irecv` in this task)*;

In [5]:
%%writefile ring_allreduce.py

import os
import torch
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh


def ring_allreduce(x: torch.Tensor, pg: dist.ProcessGroup):
    # Your code is here
    pass


def main():
    world_size = int(os.environ["WORLD_SIZE"])
    mesh = init_device_mesh("cpu", mesh_shape=(world_size,), mesh_dim_names=("dp",))

    rank = dist.get_rank()
    x = (torch.arange(4) + rank * 4).float()
    print(f"[rank {rank}] allreduce input: {x}", flush=True)

    pg = mesh.get_group("dp")
    ring_allreduce(x, pg)

    print(f"[rank {rank}] allreduce output: {x}", flush=True)

if __name__ == "__main__":
    main()

Overwriting ring_allreduce.py


In [6]:
!OMP_NUM_THREADS=1 torchrun --standalone --nnodes=1 --nproc-per-node=4 ring_allreduce.py | grep -vF 'Gloo'

[rank 3] allreduce input: tensor([12., 13., 14., 15.])
[rank 3] allreduce output: tensor([12., 13., 14., 15.])
[rank 2] allreduce input: tensor([ 8.,  9., 10., 11.])
[rank 0] allreduce input: tensor([0., 1., 2., 3.])
[rank 2] allreduce output: tensor([ 8.,  9., 10., 11.])
[rank 0] allreduce output: tensor([0., 1., 2., 3.])
[rank 1] allreduce input: tensor([4., 5., 6., 7.])
[rank 1] allreduce output: tensor([4., 5., 6., 7.])
[rank2]:[W613 13:17:16.371831570 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
[rank0]:[W613 13:17:16.373216054 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
[rank3]:[W613 13:17:16.375902104

Run two cells below (tests) after implementing `ring_allreduce` in the cell above to check your implementation:

In [7]:
%%writefile test_ring_allreduce.py

import os, torch, torch.distributed as dist, torch.distributed.distributed_c10d as c10d
from torch.distributed.device_mesh import init_device_mesh
sent = 0
def wrap(f):
    def g(*a, **k):
        global sent; sent += (a[0] if a else k["tensor"]).numel(); return f(*a, **k)
    return g
def nope(*a, **k):
    raise AssertionError("built-in collective")
w, r = int(os.environ["WORLD_SIZE"]), int(os.environ["RANK"])
pg = init_device_mesh("cpu", (w,), mesh_dim_names=("dp",)).get_group("dp")
pg2 = [dist.new_group([0, 1]), dist.new_group([2, 3])][r // 2] if w == 4 else None
dist.send = c10d.send = wrap(c10d.send)
dist.isend = c10d.isend = wrap(c10d.isend)
for m in (dist, c10d):
    for n in "all_reduce reduce broadcast all_gather all_gather_into_tensor gather scatter reduce_scatter reduce_scatter_tensor all_to_all all_to_all_single".split():
        if hasattr(m, n): setattr(m, n, nope)
from ring_allreduce import ring_allreduce
def check(b, s, c, pg):
    global sent
    ranks, g = dist.get_process_group_ranks(pg), dist.get_world_size(pg)
    x = b * s(r) + c(r)
    y = sum(b * s(i) + c(i) for i in ranks)
    ptr, old = x.data_ptr(), sent
    ring_allreduce(x, pg)
    assert x.data_ptr() == ptr and x.shape == y.shape and x.dtype == y.dtype
    assert torch.allclose(x, y, rtol=1e-5, atol=1e-7), (r, x, y)
    assert 0 < sent - old <= 2 * x.numel() * (g - 1) // g, "wrong communication pattern"
for b, s, c in [
    (torch.arange(4*w, dtype=torch.float32).reshape(w, 4) / 3 - 2, lambda i: i + 1, lambda i: i - 2),
    (torch.linspace(-1, 1, 2*w, dtype=torch.float64), lambda i: i*i + 1, lambda i: -i),
    (torch.arange(w*(w-1), dtype=torch.float32).reshape(w-1, w) / 7, lambda i: i - 1, lambda i: 2 - i),
]:
    check(b, s, c, pg)
if w == 4:
    check(torch.arange(8, dtype=torch.float32), lambda i: i + 1, lambda i: -i, pg2)
print(f"[rank {r}] Tests for Puzzle A passed!", flush=True)

Overwriting test_ring_allreduce.py


In [8]:
!bash -eo pipefail -c 'for n in 3 4; do OMP_NUM_THREADS=1 timeout 60s torchrun --standalone --nnodes=1 --nproc-per-node=$n test_ring_allreduce.py; done | grep -vE "Gloo|connected to [0-9]+ peer ranks"'

[rank0]: Traceback (most recent call last):
[rank0]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/perf_enginnering_hw_2/test_ring_allreduce.py", line 35, in <module>
[rank0]:     check(b, s, c, pg)
[rank0]:     ~~~~~^^^^^^^^^^^^^
[rank0]:   File "/home/karke/github_repos/nebius_ai_performance_engineering/perf_enginnering_hw_2/test_ring_allreduce.py", line 28, in check
[rank0]:     assert torch.allclose(x, y, rtol=1e-5, atol=1e-7), (r, x, y)
[rank0]:            ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[rank0]: AssertionError: (0, tensor([[-4.0000, -3.6667, -3.3333, -3.0000],
[rank0]:         [-2.6667, -2.3333, -2.0000, -1.6667],
[rank0]:         [-1.3333, -1.0000, -0.6667, -0.3333]]), tensor([[-15.0000, -13.0000, -11.0000,  -9.0000],
[rank0]:         [ -7.0000,  -5.0000,  -3.0000,  -1.0000],
[rank0]:         [  1.0000,   3.0000,   5.0000,   7.0000]]))
[rank2]: Traceback (most recent call last):
[rank2]:   File "/home/karke/github_repos/nebius_ai_performance_engin

## Puzzle B: Tensor Parallelism (20 points)

<img src="https://drive.google.com/uc?id=1Yjj6slJiW6hPLOca6u36gHrXtiFJfv0V" alt="tensor_parallelism.png">

In this puzzle, you'll implement a tensor parallelism strategy: sharding individual model weights across multiple ranks. Tensor parallelism can be applied to both MLP and Attention layers in transformer blocks -- we'll stick to the Transformer MLP for convenience, the non-parallel code of which would look like this:

In [9]:
import torch
import torch.nn as nn

class TransformerMLP(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        intermediate_size: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

Now, you are to implement the Tensor Parallel MLP -- you can use [Megatron-LM paper](https://arxiv.org/abs/1909.08053) as a reference, section **3. Model Parallel Transformers**.

Hints:
* Use `dist.all_reduce` for AllReduce calls;
* Make sure RNG is synchronized between the ranks for the Dropout correctness;

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


class ColumnParallelLinear(nn.Module):
    def __init__(
            self,
            input_size: int,
            output_size: int,
            tp_mesh: dist.DeviceMesh,
        ) -> None:
        super().__init__()
        # Your code is here
        assert self.weight.data.shape[1] == input_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Your code is here
        pass

class RowParallelLinear(nn.Module):
    def __init__(
            self,
            input_size: int,
            output_size: int,
            tp_mesh: dist.DeviceMesh,
        ) -> None:
        super().__init__()
        # Your code is here
        raise NotImplementedError
        assert isinstance(self.weight, torch.Tensor)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Your code is here
        raise NotImplementedError


class ParallelTransformerMLP(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        intermediate_size: int,
        tp_mesh: dist.DeviceMesh,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.fc1 = None # Your code is here
        self.act = nn.GELU()
        self.fc2 = None # Your code is here
        self.dropout = nn.Dropout(dropout)
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Your code is here
        raise NotImplementedError

Run the tests below to check your implementation:

In [11]:
import socket, torch, torch.distributed as dist, torch.multiprocessing as mp, torch.nn.functional as F
def _port():
    s = socket.socket(); s.bind(("", 0)); p = s.getsockname()[1]; s.close(); return p
def _same(a, b):
    torch.testing.assert_close(a, b, rtol=1e-5, atol=1e-5)
def _worker(rank, ws, H, M, port, split):
    dist.init_process_group("gloo", init_method=f"tcp://127.0.0.1:{port}", rank=rank, world_size=ws)
    mesh = (dist.DeviceMesh("cpu", torch.arange(ws).view(2, ws // 2), mesh_dim_names=("dp", "tp"))["tp"]
            if split else dist.DeviceMesh("cpu", list(range(ws))))
    g = mesh.get_group(); tw, tr = dist.get_world_size(g), dist.get_rank(g)
    def cat(t, d):
        xs = [torch.empty_like(t) for _ in range(tw)]
        dist.all_gather(xs, t.detach(), group=g)
        return torch.cat(xs, d)
    torch.manual_seed(10); x = torch.randn(2, 3, H, requires_grad=True)
    c = ColumnParallelLinear(H, M, mesh); assert tuple(c.weight.shape) == (M // tw, H) and "weight" in dict(c.named_parameters())
    torch.manual_seed(20 + rank); wc = torch.randn_like(c.weight); c.weight.data.copy_(wc)
    y = c(x); _same(y, F.linear(x.detach(), wc)); y.square().sum().backward()
    xr = x.detach().clone().requires_grad_(); Wc = cat(wc, 0).requires_grad_()
    F.linear(xr, Wc).square().sum().backward(); _same(x.grad, xr.grad); _same(c.weight.grad, Wc.grad.chunk(tw, 0)[tr])
    r = RowParallelLinear(M, H, mesh); assert tuple(r.weight.shape) == (H, M // tw) and "weight" in dict(r.named_parameters())
    torch.manual_seed(30 + rank); wr = torch.randn_like(r.weight); r.weight.data.copy_(wr)
    xs = torch.randn(2, 3, M // tw, requires_grad=True); z = r(xs); X, W = cat(xs, -1), cat(wr, 1).requires_grad_()
    _same(z, F.linear(X, W)); z.square().sum().backward(); Xr = X.clone().requires_grad_();
    F.linear(Xr, W).square().sum().backward(); _same(xs.grad, Xr.grad.chunk(tw, -1)[tr]); _same(r.weight.grad, W.grad.chunk(tw, 1)[tr])
    p, q = ParallelTransformerMLP(H, M, mesh, 0.5), TransformerMLP(H, M, 0.5)
    assert "weight" in dict(p.fc1.named_parameters()) and "weight" in dict(p.fc2.named_parameters())
    torch.manual_seed(40 + rank); a, b = torch.randn_like(p.fc1.weight), torch.randn_like(p.fc2.weight)
    p.fc1.weight.data.copy_(a); p.fc2.weight.data.copy_(b)
    q.fc1.weight.data.copy_(cat(a, 0)); q.fc2.weight.data.copy_(cat(b, 1))
    torch.manual_seed(50); u = torch.randn(3, 2, H, requires_grad=True); v = u.detach().clone().requires_grad_()
    torch.manual_seed(60); yp = p(u); torch.manual_seed(60); yq = q(v)
    _same(yp, yq); yp.sum().backward(); yq.sum().backward(); _same(u.grad, v.grad)
    _same(p.fc1.weight.grad, q.fc1.weight.grad.chunk(tw, 0)[tr]); _same(p.fc2.weight.grad, q.fc2.weight.grad.chunk(tw, 1)[tr])
    dist.destroy_process_group()
for ws, H, M, split in [(2, 6, 10, 0), (3, 6, 12, 0), (4, 6, 10, 1)]:
    mp.start_processes(_worker, args=(ws, H, M, _port(), split), nprocs=ws, start_method="fork", join=True)
print("All tests for Puzzle B passed!")

W0613 13:17:29.620000 14866 torch/multiprocessing/spawn.py:165] Terminating process 15741 via signal SIGTERM


ProcessRaisedException: 

-- Process 0 terminated with the following error:
Traceback (most recent call last):
  File "/home/karke/github_repos/nebius_ai_performance_engineering/.venv/lib/python3.13/site-packages/torch/multiprocessing/spawn.py", line 87, in _wrap
    fn(i, *args)
    ~~^^^^^^^^^^
  File "/tmp/ipykernel_14866/3251122800.py", line 16, in _worker
    c = ColumnParallelLinear(H, M, mesh); assert tuple(c.weight.shape) == (M // tw, H) and "weight" in dict(c.named_parameters())
  File "/tmp/ipykernel_14866/2106101115.py", line 16, in __init__
    assert self.weight.data.shape[1] == input_size
           ^^^^^^^^^^^
  File "/home/karke/github_repos/nebius_ai_performance_engineering/.venv/lib/python3.13/site-packages/torch/nn/modules/module.py", line 1968, in __getattr__
    raise AttributeError(
        f"'{type(self).__name__}' object has no attribute '{name}'"
    )
AttributeError: 'ColumnParallelLinear' object has no attribute 'weight'


## Puzzle C: Ulysses (20 points)


Sequence/Context Parallelism strategies are the parallelization schemes on the dimension of sequence length. Even though the standard Sequence Parallelism algorithm already cuts activation memory, its main role is to shard only ops like LayerNorm and Dropout. However, when training with long context lengths, temporary allgathers for the Attention layer may lead to the OOMs. There are multiple alorithms addressing this issue.

 In this puzzle, you'll implement one of the widely adopted methods -- [Ulysses](https://arxiv.org/abs/2309.14509):

<img src="https://drive.google.com/uc?id=1M-_6cYPbcbIhE-yZRSmb6UxAsoA_E6yh" alt="tensor_parallelism.png">

Finish the code of Ulyssess Attention module.
Hints:
* The inputs and outputs are sharded (SP), in this puzzle use raw `torch.Tensor` with collective operations for simplicity;
* Process group for SP and Ulysses is the same;
* Feel free to use `einops.rearrange` for the concise transpositions.
* Tests expect **causal** self-attention, feel free to use `F.scaled_dot_product_attention(..., is_causal=True)`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from einops import rearrange


class DiffAllToAll(torch.autograd.Function):
    """
    Differentiable all-to-all (across dim 0).
    """
    @staticmethod
    def forward(ctx, x: torch.Tensor, group: dist.ProcessGroup) -> torch.Tensor:
        assert x.shape[0] == dist.get_world_size(group)
        # Your code is here
        pass

    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> tuple[torch.Tensor, None]:
        # Your code is here
        pass


class UlyssesSelfAttention(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        num_heads: int,
        mesh: dist.DeviceMesh,
    ):
        super().__init__()
        assert hidden_size % num_heads == 0

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.group = mesh.get_group("sp")
        self.sp_size = dist.get_world_size(self.group)
        assert num_heads % self.sp_size == 0

        self.qkv = nn.Linear(hidden_size, 3 * hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def local_attention(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            q, k, v: [batch, seq, n_heads / sp_size, head_size]
        Returns:
            out: : [batch, seq, n_heads / sp_size, head_size]
        """
        # Your code is here
        # Hint: Use F.scaled_dot_product_attention
        pass

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            x: [batch, seq / sp_size, hidden_size]
        Returns:
            out: [batch, seq / sp_size, hidden_size]
        """
        batch, seq_local, hidden_size = x.shape
        # Your code is here:
        # 1. QKV projections
        # 2. QKV all2alls: SP into HP:
        # 3. Exact attention on full sequence, but only for the local rank's heads
        # 4. Reverse all2all: HP into SP:
        # 5. Reshape & the final projection
        pass

    def seq_par_to_head_par(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [batch, seq / sp_size, n_heads, head_size]
        Returns:
            out: [batch, seq, n_heads / sp_size, head_size]
        """
        # Your code is here
        pass

    def head_par_to_seq_par(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [batch, seq, n_heads / sp_size, head_size]
        Returns:
            out: [batch, seq / sp_size, n_heads, head_size]
        """
        # Your code is here
        pass

Run the tests below to check your implementation:

In [ ]:
import os, tempfile, torch, torch.nn.functional as F
import torch.distributed as dist, torch.multiprocessing as mp
from torch.distributed.device_mesh import init_device_mesh
def _ulysses_tests(rank, world, url):
    torch.set_num_threads(1); dist.init_process_group("gloo", init_method=url, rank=rank, world_size=world)
    try:
        mesh = init_device_mesh("cpu", (world,), mesh_dim_names=("sp",)); group = mesh.get_group("sp")
        base = torch.arange(world * world * 3., dtype=torch.float32).reshape(world, world, 3)
        x = (base + 100 * rank).requires_grad_()
        y = DiffAllToAll.apply(x, group)
        assert torch.equal(y, torch.stack([base[rank] + 100 * r for r in range(world)])), \
            f"rank {rank}, world {world}: DiffAllToAll forward/order is wrong"
        (y * (rank + 1)).sum().backward()
        assert torch.equal(x.grad, torch.arange(1., world + 1).view(world, 1, 1).expand_as(x)), \
            f"rank {rank}, world {world}: DiffAllToAll backward/grad routing is wrong"
        dist.barrier()
        m = UlyssesSelfAttention(8, 4, mesh)
        z = (torch.randn(2, 3, 4, 2) + rank).requires_grad_()
        assert torch.allclose(m.head_par_to_seq_par(m.seq_par_to_head_par(z)), z), \
            f"rank {rank}, world {world}: SP<->HP reshuffle is not an inverse; check all-to-all layout/contiguity"
        dist.barrier(); torch.manual_seed(0); S = 3 * world; full = torch.randn(2, S, 8)
        x = full.chunk(world, 1)[rank].clone().requires_grad_()
        m = UlyssesSelfAttention(8, 4, mesh)
        m_ref = UlyssesSelfAttention(8, 4, mesh)
        m_ref.load_state_dict(m.state_dict())
        got = m(x)
        full = full.clone().requires_grad_()
        q, k, v = m_ref.qkv(full).view(2, S, 3, 4, 2).unbind(2)
        ref = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=True)
        ref = m_ref.out_proj(ref.transpose(1, 2).reshape(2, S, 8))
        assert torch.allclose(got, ref.chunk(world, 1)[rank], atol=2e-5), \
            f"rank {rank}, world {world}: forward output differs from full causal attention"
        got.square().sum().backward(); ref.square().sum().backward()
        assert torch.allclose(x.grad, full.grad.chunk(world, 1)[rank], atol=2e-5), \
            f"rank {rank}, world {world}: input gradients differ from full causal attention"
        for p, p_ref in zip(m.parameters(), m_ref.parameters()):
            grad = p.grad.detach().clone(); dist.all_reduce(grad, group=group)
            assert torch.allclose(grad, p_ref.grad, atol=2e-5), \
                f"rank {rank}, world {world}: replicated parameter gradients differ from full causal attention"
    finally:
        dist.destroy_process_group()
for world in (2, 4):
    with tempfile.TemporaryDirectory() as d:
        mp.start_processes(_ulysses_tests, args=(world, "file://" + os.path.join(d, "pg")),
                           nprocs=world, start_method="fork", join=True)
print("All tests for Puzzle C passed!")

## Puzzle D: Ring Attention (30 points)

<img src="https://drive.google.com/uc?id=1xYROFFSwGZlPeKqsd40wX1h2waG1sJx1" alt="ring_attention.png">

Modern LLMs may support extremely long contexts! However, fitting that context into the devices introduces a bunch of engineering challenges. In this puzzle, you will implement one of the very elegant algorithms for long context training: [Ring Attention](https://arxiv.org/abs/2310.01889).

Learning resources breaking down the ring attention algorithm besides the original paper:
* [GPU MODE Ring Attention Lecture](https://youtu.be/ws7angQYIxI);
* [Ultra-Long Sequence Parallelism](https://huggingface.co/blog/exploding-gradients/ulysses-ring-attention);
* [Ring Attention Explained](https://coconut-mode.com/posts/ring-attention/).

Some simplifications:
* Compute bidirectional attention (no attention mask);
* Assume the full sequence length is divisible by the world size;
* We only implement Ring Attention on its own, without QKV linear projections, output projection, or tensor-parallel comms.
* We only implement on Ring Attention forward.

Some hints:
* Remember about the scale $\sqrt{d}$ and the batched computation. That is, the actual attention formula:
$$A=\operatorname{Softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V,\quad Q,K,V,A\in\mathbb{R}^{b\times h\times s\times d}$$
* Use `dist.batch_isend_irecv` in `start_kv_rotate` to schedule all p2p comms as a single collective operation.

To begin with, we'll implement the comms part -- rotation of KVs. We'd need to primitives:
* `start_kv_rotate` that will schedule the comms *(to be overlapped with the compute)*;
* `wait_all` that can be used by the end of the steps to make sure KVs have been rotated.

In [ ]:
import torch
import torch.distributed as dist

def start_kv_rotate(
    k: torch.Tensor,
    v: torch.Tensor,
    k_recv: torch.Tensor,
    v_recv: torch.Tensor,
    src_rank: int,
    dst_rank: int,
    pg: dist.ProcessGroup,
) -> list[dist.Work]:
    """
    Start one async KV ring-rotation step.
    Each rank sends its current (k, v) to dst_rank and receives next (k, v) from
    src_rank into (k_recv, v_recv).
    """
    # Your code is here: use `dist.batch_isend_irecv`, `dist.P2POp`,
    # `dist.isend`, and `dist.irecv`
    raise NotImplementedError


def wait_all(reqs: list[dist.Work]) -> None:
    """Wait for all async dist requests to complete"""
    # Your code is here
    raise NotImplementedError

Run the tests below to check your implementation of auxiliary functions:

In [ ]:
import datetime as _dt, tempfile, torch.multiprocessing as mp
class _W:
    def __init__(self): self.n = 0
    def wait(self): self.n += 1
_ws = [_W(), _W()]; wait_all(_ws); assert [w.n for w in _ws] == [1, 1]
def _test_kv(rank, path):
    dist.init_process_group("gloo", init_method=f"file://{path}", rank=rank, world_size=3, timeout=_dt.timedelta(seconds=20))
    k = torch.full((2, 3), rank + .25); v = k + 10; kr, vr = torch.empty_like(k), torch.empty_like(v)
    reqs = start_kv_rotate(k, v, kr, vr, (rank - 1) % 3, (rank + 1) % 3, dist.group.WORLD)
    assert len(reqs) == 4; wait_all(reqs)
    torch.testing.assert_close(kr, torch.full_like(k, (rank - 1) % 3 + .25))
    torch.testing.assert_close(vr, torch.full_like(v, (rank - 1) % 3 + 10.25))
    dist.destroy_process_group()
mp.start_processes(_test_kv, args=(tempfile.mktemp(),), nprocs=3, start_method="fork")
print("Comms tests for Puzzle D passed!")

Now, implement the Ring Attention algorithm using the [Online Softmax idea](https://arxiv.org/abs/1805.02867).

Hints:
* You follow the approach from the lectures (slide 58) -- storing the unnormalized numerator and denominator separately -- it would make the code more concise.
* Alternatively, you may implement inner loop as in Flash Attention ([paper](https://arxiv.org/pdf/2205.14135) -- see `Algorithm 1`) -- this may require slightly more code, but would update the result online instead of keeping unnormalized numerator.
* Reminder on attention simplifications: forward-only, bidirectional (unmasked).

In [ ]:
import math
import torch
import torch.distributed as dist

def ring_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    pg: dist.ProcessGroup,
):
    """
    distributed exact unmasked self-attention over a context
    sharded across pg group.
    ---
    inputs:
        q, k, v: [b, h, s_local, d]
    outputs:
        a: [b, h, s_local, d]
    ----
    * b: batch size
    * h: number of heads
    * s_local: local sequence length (s_local == s_global / world_size)
    * d: head dimension
    """
    rank = dist.get_rank(pg)
    world_size = dist.get_world_size(pg)
    src_rank = (rank - 1 + world_size) % world_size
    dst_rank = (rank + 1) % world_size

    k_recv = torch.empty_like(k)
    v_recv = torch.empty_like(v)

    # Your code is here: initialize the buffers

    for step in range(world_size):
        # Your code is here: schedule the comms (if necessary)

        # Your code is here: update the running statistics

        # Your code is here: wait for the comms and update data (if necessary)

        raise NotImplementedError

    # Your code is here: return the result

Run the tests to check the implementation:

In [ ]:
import datetime as _dt, tempfile, torch.multiprocessing as mp, torch.nn.functional as F
def _test_attn(rank, path, mult):
    dist.init_process_group("gloo", init_method=f"file://{path}", rank=rank, world_size=3, timeout=_dt.timedelta(seconds=20))
    global start_kv_rotate
    old, seen = start_kv_rotate, []
    def spy(*a, **kw): seen.append(1); return old(*a, **kw)
    start_kv_rotate = spy
    g = torch.Generator().manual_seed(713 + int(mult)); B,H,S,D,W = 2,2,3,5,3; sl = slice(rank*S, (rank+1)*S)
    Q = torch.randn(B,H,S*W,D, generator=g) * mult; K = torch.randn(B,H,S*W,D, generator=g) * mult; V = torch.randn(B,H,S*W,D, generator=g)
    y = ring_attention(Q[:,:,sl].clone(), K[:,:,sl].clone(), V[:,:,sl].clone(), dist.group.WORLD)
    torch.testing.assert_close(y.float(), F.scaled_dot_product_attention(Q[:,:,sl].float(), K.float(), V.float()), rtol=2e-4, atol=2e-4)
    assert len(seen) == 2
    dist.destroy_process_group()
for _m in (1, 20): mp.start_processes(_test_attn, args=(tempfile.mktemp(), _m), nprocs=3, start_method="fork")
print("Attention tests for Puzzle D passed!")

## Puzzle E: Zero Bubble Pipelining (10 points)

<img src="https://drive.google.com/uc?id=1rFjOYW5AXXP0u4NoKXbGM1k_cNUmbV1J" alt="zero_bubble_pipeline_parallelism.png">

Pipeline parallelism partitions a model along its depth dimension, typically by assigning the consecutive groups of layers to different devices. Compared other training parallelisms, PP requirements to the network bandwidth are typically minimal, as we only send a portion of activations from one device to another! However, pipelining may introduce some idle time when devices are not utilized, called bubbles. The main goal of pipeline-parallel algorithms is to minimise pipeline bubbles: periods when some stages are idle.

In this puzzle, you will implement a [ZB-H2-style Zero Bubble Pipeline](https://arxiv.org/abs/2401.10241) schedule from the paper. Zero Bubble pipelining splits the backward computation for each microbatch into two parts:
* Gradient with respect to the input;
* Gradient with respect to the weights -- and this stage is not on the critical path, so it may be delayed.

We'll rely on some idealized assumptions:
1. `forward time (F) == backward weight time (W) == backward input time (B)`, communication time is ignored.
2. $m \geqslant 2n$, where $n$ is the number of processes, and $m$ is the number of microbatches.

It turns out that in this case you have the optimal schedule in closed form! *(there are actually a lot of possible solutions correct under those assumptions)*

Hints:
* Each stage $i$ starts with $i$ idle slots.
* Each stage has $3m$ non-idle operations.
* Trailing idle slots may be omitted.
* Make sure your pipelining schedule is a correct pipelining algorithm (e.g., `F{i}` occurs before `B{i}`).
* Look at the print example below.

In [ ]:
def zb_h2_schedule(n: int, m: int) -> list[list[str]]:
    """
    Return Zero Bubble ZB-H2 schedule as rows of tokens.
    ----
    inputs:
        n: number of devices
        m: number of microbatches

    returns:
        out: list n timeline rows. out[i][j] is the operation run by stage i
        at time j. each operation is a string like: "F1", "B2", "W123",
        or "" for idle compute.
        Operations meanings:
          "F{j}": forward computation for microbatch j
          "B{j}": backward-input computation for microbatch j
          "W{j}": backward-weight computation for microbatch j
          "": idle compute

    """
    if m < 2 * n:
        raise ValueError("This closed-form ZB-H2 schedule assumes m >= 2n")

    schedule = []

    for stage in range(n):
        row = [""] * stage

        # Your code is here

        schedule.append(row)

    return schedule

Run the print example to see if you can reproduce the diagram from the paper:

In [ ]:
def print_schedule(rows, width=4):
    reset = "\033[0m"
    colors = {
        "F": "\033[30;48;5;153m", "B": "\033[30;48;5;223m",
        "W": "\033[30;48;5;194m", "":  "\033[30;48;5;255m",
    }
    ncols = max(len(r) for r in rows)
    width = max(3, max((len(str(cell)) for row in rows for cell in row),
                       default=1) + 2)
    for row in rows:
        row = row + [""] * (ncols - len(row))
        print("".join(
            f"{colors.get(cell[:1], colors[''])}{cell:^{width}}{reset}"
            for cell in row
        ))

print_schedule(zb_h2_schedule(n=4, m=8))

Run the tests below to check your implementation:

In [ ]:
import random
import re
TOK = re.compile(r"([FBW])([1-9]\d*)\Z")
def assert_zb_h2(make):
    rng = random.Random(0)
    for n in [rng.randint(2, 100) for _ in range(10)]:
        try:
            make(n, 2 * n - 1)
        except ValueError:
            pass
        else:
            raise AssertionError((n, 2 * n - 1, "should reject m < 2n"))
    cases = [(n, m) for n in [rng.randint(1, 10) for _ in range(10)] for m in range(2 * n, 2 * n + 10)]
    for n, m in cases:
        rows = make(n, m)
        assert isinstance(rows, list) and len(rows) == n, (n, m, "bad row count")
        pos = {}
        for s, row in enumerate(rows):
            assert isinstance(row, list), (n, m, s, "row is not list")
            assert all(isinstance(x, str) for x in row), (n, m, s, "non-string token")
            active = [t for t, x in enumerate(row) if x]
            assert active == list(range(s, s + 3 * m)), (n, m, s, "bubble or wrong skew")
            for t, x in enumerate(row):
                if not x:
                    continue
                z = TOK.fullmatch(x)
                assert z, (n, m, s, t, "bad token", x)
                op, mb = z.group(1), int(z.group(2))
                assert 1 <= mb <= m, (n, m, s, t, "bad microbatch", x)
                assert (s, op, mb) not in pos, (n, m, s, t, "duplicate", x)
                pos[s, op, mb] = t
        assert len(pos) == 3 * n * m, (n, m, "missing or extra ops")
        for s in range(n):
            for mb in range(1, m + 1):
                f, b, w = (pos[s, op, mb] for op in "FBW")
                assert f < b < w, (n, m, s, mb, "local order: F -> B -> W")
                if s > 0:
                    assert pos[s - 1, "F", mb] < f, (n, m, s, mb, "forward pipeline dependency")
                if s + 1 < n:
                    assert pos[s + 1, "B", mb] < b, (n, m, s, mb, "backward-input pipeline dependency")
            # ZB-H2 uses FIFO F and FIFO B; W is intentionally not required FIFO
            assert all(pos[s, "F", i] < pos[s, "F", i + 1] for i in range(1, m)), (
                n, m, s, "F not FIFO"
            )
            assert all(pos[s, "B", i] < pos[s, "B", i + 1] for i in range(1, m)), (
                n, m, s, "B not FIFO"
            )
    print("All tests for Puzzle E passed!")
assert_zb_h2(zb_h2_schedule)